In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import Any, cast

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
import qiskit.primitives as primitives
from qiskit.quantum_info import SparsePauliOp

# ============================================
# Configuration
# ============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Re = 100.0
n_collocation = 500
epochs = 500
lr = 1e-3

# ============================================
# Collocation Points
# ============================================
def sample_points(n):
    x = torch.rand(n, 1)
    y = torch.rand(n, 1)
    return torch.cat([x, y], dim=1).to(device)

# ============================================
# Classical PINN
# ============================================
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 3)  # u,v,p
        )

    def forward(self, x):
        return self.net(x)

# ============================================
# Quantum Layer
# ============================================
class QLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.randn(6))
        self.params = ParameterVector("θ", 6)

        qc = QuantumCircuit(2)
        qc.ry(self.params[0], 0)
        qc.ry(self.params[1], 1)
        qc.cz(0,1)
        qc.ry(self.params[2], 0)
        qc.ry(self.params[3], 1)
        qc.cz(0,1)
        qc.ry(self.params[4], 0)
        qc.ry(self.params[5], 1)
        self.circuit = qc
        self.observable = SparsePauliOp.from_list([("ZZ", 1.0)])
        self.estimator = primitives.StatevectorEstimator()

    def forward(self, x):
        outputs = []
        for _ in x:
            theta_vals = self.theta.detach().cpu().numpy()
            pub = (self.circuit, self.observable,)
            job = self.estimator.run([pub])
            pub_result = job.result()[0]
            evs = np.asarray(cast(Any, pub_result.data).evs)
            result = evs.reshape(-1)[0]
            outputs.append([result])
            outputs.append([result])
        return torch.tensor(outputs, dtype=torch.float32).to(device)

# ============================================
# QPINN Model
# ============================================
class QPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre = nn.Linear(2, 2)
        self.quantum = QLayer()
        self.post = nn.Linear(1, 3)

    def forward(self, x):
        x = torch.tanh(self.pre(x))
        q_out = self.quantum(x)
        return self.post(q_out)

# ============================================
# NSE Residual
# ============================================
def residual(model, xy):
    xy.requires_grad_(True)
    out = model(xy)
    u = out[:,0:1]
    v = out[:,1:2]
    p = out[:,2:3]

    grads = torch.ones_like(u)

    u_x = torch.autograd.grad(u, xy, grads, create_graph=True)[0][:,0:1]
    u_y = torch.autograd.grad(u, xy, grads, create_graph=True)[0][:,1:2]
    v_x = torch.autograd.grad(v, xy, grads, create_graph=True)[0][:,0:1]
    v_y = torch.autograd.grad(v, xy, grads, create_graph=True)[0][:,1:2]
    p_x = torch.autograd.grad(p, xy, grads, create_graph=True)[0][:,0:1]
    p_y = torch.autograd.grad(p, xy, grads, create_graph=True)[0][:,1:2]

    u_xx = torch.autograd.grad(u_x, xy, grads, create_graph=True)[0][:,0:1]
    u_yy = torch.autograd.grad(u_y, xy, grads, create_graph=True)[0][:,1:2]
    v_xx = torch.autograd.grad(v_x, xy, grads, create_graph=True)[0][:,0:1]
    v_yy = torch.autograd.grad(v_y, xy, grads, create_graph=True)[0][:,1:2]

    R_u = u*u_x + v*u_y + p_x - (1/Re)*(u_xx + u_yy)
    R_v = u*v_x + v*v_y + p_y - (1/Re)*(v_xx + v_yy)
    R_c = u_x + v_y

    return torch.mean(R_u**2 + R_v**2 + R_c**2)

# ============================================
# Train Function
# ============================================
def train(model):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        opt.zero_grad()
        xy = sample_points(n_collocation)
        loss = residual(model, xy)
        loss.backward()
        opt.step()
        if epoch % 100 == 0:
            print(f"Epoch {epoch}: {loss.item():.6f}")
    return model

# ============================================
# Run Comparison
# ============================================
print("\nTraining Classical PINN...")
pinn = train(PINN().to(device))

print("\nTraining Quantum PINN...")
qpinn = train(QPINN().to(device))

# Evaluation
xy_test = sample_points(1000)
pinn_res = residual(pinn, xy_test).item()
qpinn_res = residual(qpinn, xy_test).item()

print("\n=============================")
print("Residual Norm Comparison")
print("=============================")
print(f"PINN Residual Norm :  {pinn_res:.6e}")
print(f"QPINN Residual Norm:  {qpinn_res:.6e}")
print("=============================")